In [23]:
import matplotlib.pyplot as plt
import os
import yaml
from pprint import pprint
import numpy as np



def open_yaml(yaml_path):
    
    with open(yaml_path) as f:
        loadedConfig = yaml.load(f, Loader=yaml.FullLoader)
    return loadedConfig

def plotPerData(plot_save_path,data,label,do_plot=True):
    
    f_results = {
        f'denoising_dsvdd_{i}':[] for i in range(1,9)
    }
    f_results['ocsvm']= []
    f_results['vanilla_dsvdd'] = []
    f_results['smooth_dsvdd'] = []
    
    roc_results= {
        f'denoising_dsvdd_{i}':[] for i in range(1,9)
    } 
    roc_results['ocsvm']= []
    roc_results['vanilla_dsvdd'] = []
    roc_results['smooth_dsvdd'] = []
    
    ap_results = {
        f'denoising_dsvdd_{i}':[] for i in range(1,9)
    }
    ap_results['ocsvm']= []
    ap_results['vanilla_dsvdd'] = []
    ap_results['smooth_dsvdd'] = []
    
    
    
    ocsvm_cnt = 0
    ocsvm_motherPath = f'./history/ocsvm_grad_test_3/{data}/normal_label_{label}/'
    for eachCase in os.listdir(ocsvm_motherPath):
        # print(eachCase)
        try:
            eachYamlPath = os.path.join(ocsvm_motherPath,eachCase,'configs/resultConfig.yaml')
            
        
            openedYaml = open_yaml(eachYamlPath)
            
            f_results['ocsvm'].append(float(openedYaml['testResult']['f1']))
            roc_results['ocsvm'].append(float(openedYaml['testResult']['rocAuc']))
            ap_results['ocsvm'].append(float(openedYaml['testResult']['averagePrecision']))
            
            ocsvm_cnt +=1
            
            if ocsvm_cnt == 10:
                break
            
        except:
            
            pass
            
            
    vanilla_dsvdd_cnt = 0
    vanilla_dsvdd_motherPath = f'./history/vanilla_dsvdd_grad_test/{data}/normal_label_{label}/'# ocsvm
    for eachCase in os.listdir(vanilla_dsvdd_motherPath):
        
        try:
            
            eachYamlPath = os.path.join(vanilla_dsvdd_motherPath,eachCase,'configs/resultConfig.yaml')
        
            openedYaml = open_yaml(eachYamlPath)
            # pprint(openedYaml)
            f_results['vanilla_dsvdd'].append(float(openedYaml['testResult']['f1_score']))
            roc_results['vanilla_dsvdd'].append(float(openedYaml['testResult']['roc_auc']))
            ap_results['vanilla_dsvdd'].append(float(openedYaml['testResult']['average_precision']))
            
            vanilla_dsvdd_cnt += 1
            
            if vanilla_dsvdd_cnt == 10:
                break
        
        except:
            
            pass
            
        
    smooth_dsvdd_cnt = 0
    smooth_dsvdd_motherPath = f'./history/smoothed_dsvdd_grad_test/{data}/normal_label_{label}/'# ocsvm
    for eachCase in os.listdir(smooth_dsvdd_motherPath):
        
        try:
    
            eachYamlPath = os.path.join(smooth_dsvdd_motherPath,eachCase,'configs/resultConfig.yaml')
        
            openedYaml = open_yaml(eachYamlPath)
        
            f_results['smooth_dsvdd'].append(float(openedYaml['testResult']['f1_score']))
            roc_results['smooth_dsvdd'].append(float(openedYaml['testResult']['roc_auc']))
            ap_results['smooth_dsvdd'].append(float(openedYaml['testResult']['average_precision']))
            
            smooth_dsvdd_cnt += 1
            
            if smooth_dsvdd_cnt == 10:
                break
            
        except:
            
            pass
            
            
    denosing_cnt_lst = []
    denoising_motherPath = f'./history/denoising_dsvdd_grad_test/{data}/normal_label_{label}/'    
    for noiseRatio in range(1,9):
        
        eachMotherPath = os.path.join(denoising_motherPath,'noise_'+str(round(0.1*noiseRatio,1)))
        
        denoising_cnt = 0
        
        for eachCase in os.listdir(eachMotherPath):
            
            try:
                eachYamlPath = os.path.join(eachMotherPath,eachCase,'configs/resultConfig.yaml')
            
                openedYaml = open_yaml(eachYamlPath)
            
                f_results[f'denoising_dsvdd_{noiseRatio}'].append(float(openedYaml['testResult']['f1_score']))
                roc_results[f'denoising_dsvdd_{noiseRatio}'].append(float(openedYaml['testResult']['roc_auc']))
                ap_results[f'denoising_dsvdd_{noiseRatio}'].append(float(openedYaml['testResult']['average_precision']))
                
                denoising_cnt += 1
                
                if denoising_cnt == 10:
                    break
                
            except:
                
                pass
        
        denosing_cnt_lst.append(denoising_cnt)
                
    
    if ocsvm_cnt < 8:
        print(f'{data,label}ocsvm_cnt must be 10 but got {ocsvm_cnt}')
        raise Exception(
            f'ocsvm_cnt must be 10 but got {ocsvm_cnt}'
        )
    
    if vanilla_dsvdd_cnt < 8:
        
        print(f'{data,label}vanilla_cnt must be 10 but got {vanilla_dsvdd_cnt}')
        raise Exception(
            f'vanilla_cnt must be 10 but got {vanilla_dsvdd_cnt}'
        )
        
    if smooth_dsvdd_cnt < 8:
        
        print(f'{data,label}smooth_cnt must be 10 but got {smooth_dsvdd_cnt}')
        raise Exception(
            
            f'smooth_cnt must be 10 but got {smooth_dsvdd_cnt}'
        )
        
    for idx,each_cnt in enumerate(denosing_cnt_lst):
        
        if each_cnt <8:
            print(f'{data,label}noise ratio {idx+1} cnt must be 10 but got {each_cnt}')
            raise Exception(
                f'noise ratio {idx+1} cnt must be 10 but got {each_cnt}'
            )
            
            
    order_lst = [
        'ocsvm',
        'vanilla_dsvdd',
        'smooth_dsvdd',
    ]
    order_lst.extend([
        f'denoising_dsvdd_{noiseRatio}' for noiseRatio in range(1,9)
    ])
    
    result_mean_f = {}
    result_mean_roc = {}
    result_mean_ap = {}
    
    result_std_f = {}
    result_std_roc = {}
    result_std_ap = {}
    
    best_name_f = ''
    best_name_roc = ''
    best_name_ap = ''
    best_score_f = -1
    best_score_roc = -1
    best_score_ap = -1
    
    for order in order_lst:
        
        if np.mean(f_results[order])>= best_score_f:
            best_name_f = order
            best_score_f = np.mean(f_results[order])
            
        if np.mean(roc_results[order]) >= best_score_roc:
            best_name_roc = order
            best_score_roc = np.mean(roc_results[order])
            
        if np.mean(ap_results[order]) >= best_score_ap:
            best_name_ap = order
            best_score_ap = np.mean(ap_results[order])
        
        
        result_mean_f[order] = np.mean(f_results[order])
        result_mean_roc[order] = np.mean(roc_results[order])
        result_mean_ap[order] = np.mean(ap_results[order])
        
        result_std_f[order] = np.std(f_results[order],ddof=1)
        result_std_roc[order] = np.std(roc_results[order],ddof=1)
        result_std_ap[order] = np.std(ap_results[order],ddof=1)
        
    if do_plot:
        f_result_show = []
        roc_result_show = []
        ap_result_show = []
        
        for order in order_lst:
            # print(order)
            f_result_show.append(f_results[order])
            roc_result_show.append(roc_results[order])
            ap_result_show.append(ap_results[order])
            
        # print(f_result_show)
        plt.boxplot(
            f_result_show
        )
        plt.xticks([i+1 for i in range(len(order_lst))],order_lst,rotation=45)
        plt.savefig(
            os.path.join(
                plot_save_path,
                'f1_score.png'
            ),
            dpi = 600
            
        )
        plt.close()
        plt.cla()
        plt.clf()
        
        
        plt.boxplot(
            roc_result_show
        )
        plt.xticks([i+1 for i in range(len(order_lst))],order_lst,rotation=45)
        plt.savefig(
            os.path.join(
                plot_save_path,
                'roc_auc_score.png'
            ),
            dpi = 600
        )
        plt.close()
        plt.cla()
        plt.clf()
        
        plt.boxplot(
            ap_result_show
        )
        plt.xticks([i+1 for i in range(len(order_lst))],order_lst,rotation=45)
        plt.savefig(
            os.path.join(
                plot_save_path,
                'average_precision.png'
            ),
            dpi = 600
        )
        plt.close()
        plt.cla()
        plt.clf()
    
    return f_results, roc_results, ap_results,result_mean_f,result_mean_roc,result_mean_ap,result_std_f,result_std_roc,result_std_ap,best_name_f,best_name_roc,best_name_ap,best_score_f,best_score_roc,best_score_ap
    
        
import pandas as pd

data_lst = [
    'mnist_1',
    'mnist_2',
    'mnist_3',
    'cifar_1',
    'cifar_2',
    'cifar_3'
]

label_lst = [str(i) for i in range(0,9+1)]

total_result = []

order_lst = [
        'ocsvm',
        'vanilla_dsvdd',
        'smooth_dsvdd',
    ]
order_lst.extend([
    f'denoising_dsvdd_{noiseRatio}' for noiseRatio in range(1,9)
])

df_lst = []
for data in data_lst:
    
    for label in label_lst:
        
        plot_save_path = f'./result_save_path/{data}_{label}'
        os.makedirs(plot_save_path)
        
        f_results, roc_results, ap_results,result_mean_f,result_mean_roc,result_mean_ap,result_std_f,result_std_roc,result_std_ap,best_name_f,best_name_roc,best_name_ap,best_score_f,best_score_roc,best_score_ap = plotPerData(plot_save_path,data,label,do_plot=True)
        
        each_total_result = [
            data,
            label,
            best_name_f,
            best_name_roc,
            best_name_ap,
            best_score_f,
            best_score_roc,
            best_score_ap
        ]
        
        for order in order_lst:
            
            each_total_result.extend(
                [
                    result_mean_f[order],
                    result_std_f[order],
                    result_mean_roc[order],
                    result_std_roc[order],
                    result_mean_ap[order],
                    result_std_ap[order]
                ]
            )
            
        total_result.append(each_total_result)

column_lst = [
    'data_type',
    'normal_label',
    'model achieved highest f1 score(mean)',
    'model achieved highest roc auc score(mean)',
    'model achieved highest average precision score(mean)',
    'highest f1 score(mean)',
    'highest f1 score(mean)',
    'highest f1 score(mean)'
]
for order in order_lst:
    column_lst.extend(
        [
            f'{order}_f1_score_mean',
            f'{order}_f1_score_std',
            f'{order}_roc_auc_mean',
            f'{order}_roc_auc_std',
            f'{order}_average_precision_mean',
            f'{order}_average_precision_std',    
        ]
        
    )
    
total_result = pd.DataFrame(
    total_result,
    columns= column_lst

)
        
        
total_result.to_csv(
    './total_result_mnist_cifar.csv',
    index=False
)
        
print('mission complete!!!')        
        
    
        
        



    
        
        
    
    
    
    
    





mission complete!!!


<Figure size 640x480 with 0 Axes>

In [16]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

test_x = np.random.rand(1000)
test_y = np.random.randint(0,2,size=(1000))

print(test_x.shape)
print(test_y.shape)

before_score_roc = roc_auc_score(test_y,test_x)
before_score_ap = average_precision_score(test_y,test_x)
# print(test_x)
test_x_new = (test_x-min(test_x))/(max(test_x)-min(test_x))
# print(test_x_new)
after_score_roc = roc_auc_score(test_y,test_x_new)
after_score_ap = average_precision_score(test_y,test_x_new)

print(before_score_roc,before_score_ap)
print(after_score_roc,after_score_ap)



(1000,)
(1000,)
0.5223474886137661 0.48263084822583036
0.5223474886137661 0.48263084822583036


In [25]:
import matplotlib.pyplot as plt
import os
import yaml
from pprint import pprint
import numpy as np



def open_yaml(yaml_path):
    
    with open(yaml_path) as f:
        loadedConfig = yaml.load(f, Loader=yaml.FullLoader)
    return loadedConfig

def plotPerData(plot_save_path,data,label,do_plot=True):
    
    f_results = {
        f'denoising_dsvdd_{i}':[] for i in range(1,9)
    }
    f_results['ocsvm']= []
    f_results['vanilla_dsvdd'] = []
    f_results['smooth_dsvdd'] = []
    
    roc_results= {
        f'denoising_dsvdd_{i}':[] for i in range(1,9)
    } 
    roc_results['ocsvm']= []
    roc_results['vanilla_dsvdd'] = []
    roc_results['smooth_dsvdd'] = []
    
    ap_results = {
        f'denoising_dsvdd_{i}':[] for i in range(1,9)
    }
    ap_results['ocsvm']= []
    ap_results['vanilla_dsvdd'] = []
    ap_results['smooth_dsvdd'] = []
    
    ocsvm_cnt = 0
    ocsvm_motherPath = f'./history/ocsvm_grad_test_testbed/{data}_noiseRatio_{label}/'
    for eachCase in os.listdir(ocsvm_motherPath):
        # print(eachCase)
        try:
            eachYamlPath = os.path.join(ocsvm_motherPath,eachCase,'configs/resultConfig.yaml')
            
        
            openedYaml = open_yaml(eachYamlPath)
            
            f_results['ocsvm'].append(float(openedYaml['testResult']['f1']))
            roc_results['ocsvm'].append(float(openedYaml['testResult']['rocAuc']))
            ap_results['ocsvm'].append(float(openedYaml['testResult']['averagePrecision']))
            
            ocsvm_cnt +=1
            
            if ocsvm_cnt == 10:
                break
            
        except:
            
            pass
            
    
    vanilla_dsvdd_cnt = 0
    vanilla_dsvdd_motherPath = f'./history/vanilla_dsvdd_grad_test_testbed/{data}_noiseRatio_{label}/'# ocsvm
    for eachCase in os.listdir(vanilla_dsvdd_motherPath):
        
        try:
            
            eachYamlPath = os.path.join(vanilla_dsvdd_motherPath,eachCase,'configs/resultConfig.yaml')
        
            openedYaml = open_yaml(eachYamlPath)
            # pprint(openedYaml)
            f_results['vanilla_dsvdd'].append(float(openedYaml['testResult']['f1_score']))
            roc_results['vanilla_dsvdd'].append(float(openedYaml['testResult']['roc_auc']))
            ap_results['vanilla_dsvdd'].append(float(openedYaml['testResult']['average_precision']))
            
            vanilla_dsvdd_cnt += 1
            
            if vanilla_dsvdd_cnt == 10:
                break
        
        except:
            
            pass
            
    smooth_dsvdd_cnt = 0
    smooth_dsvdd_motherPath = f'./history/smoothed_dsvdd_grad_test_testbed/{data}_noiseRatio_{label}/'# ocsvm
    for eachCase in os.listdir(smooth_dsvdd_motherPath):
        
        try:
    
            eachYamlPath = os.path.join(smooth_dsvdd_motherPath,eachCase,'configs/resultConfig.yaml')
        
            openedYaml = open_yaml(eachYamlPath)
        
            f_results['smooth_dsvdd'].append(float(openedYaml['testResult']['f1_score']))
            roc_results['smooth_dsvdd'].append(float(openedYaml['testResult']['roc_auc']))
            ap_results['smooth_dsvdd'].append(float(openedYaml['testResult']['average_precision']))
            
            smooth_dsvdd_cnt += 1
            
            if smooth_dsvdd_cnt == 10:
                break
            
        except:
            
            pass
            
    denosing_cnt_lst = []
    denoising_motherPath = f'./history/denoising_dsvdd_grad_test_3/{data}_noiseRatio_{label}/'    
    for noiseRatio in range(1,9):
        
        eachMotherPath = os.path.join(denoising_motherPath,'noise_'+str(round(0.1*noiseRatio,1)))
        
        denoising_cnt = 0
        
        for eachCase in os.listdir(eachMotherPath):
            
            try:
                eachYamlPath = os.path.join(eachMotherPath,eachCase,'configs/resultConfig.yaml')
            
                openedYaml = open_yaml(eachYamlPath)
            
                f_results[f'denoising_dsvdd_{noiseRatio}'].append(float(openedYaml['testResult']['f1_score']))
                roc_results[f'denoising_dsvdd_{noiseRatio}'].append(float(openedYaml['testResult']['roc_auc']))
                ap_results[f'denoising_dsvdd_{noiseRatio}'].append(float(openedYaml['testResult']['average_precision']))
                
                denoising_cnt += 1
                
                if denoising_cnt == 10:
                    break
                
            except:
                
                pass
        
        denosing_cnt_lst.append(denoising_cnt)
                
    
    if ocsvm_cnt != 10:
        print(f'{data,label}ocsvm_cnt must be 10 but got {ocsvm_cnt}')
        raise Exception(
            f'ocsvm_cnt must be 10 but got {ocsvm_cnt}'
        )
    
    if vanilla_dsvdd_cnt != 10:
        
        print(f'{data,label}vanilla_cnt must be 10 but got {vanilla_dsvdd_cnt}')
        raise Exception(
            f'vanilla_cnt must be 10 but got {vanilla_dsvdd_cnt}'
        )
        
    if smooth_dsvdd_cnt != 10:
        
        print(f'{data,label}smooth_cnt must be 10 but got {smooth_dsvdd_cnt}')
        raise Exception(
            
            f'smooth_cnt must be 10 but got {smooth_dsvdd_cnt}'
        )
        
    for idx,each_cnt in enumerate(denosing_cnt_lst):
        
        if each_cnt != 10:
            print(f'{data,label}noise ratio {idx+1} cnt must be 10 but got {each_cnt}')
            raise Exception(
                f'noise ratio {idx+1} cnt must be 10 but got {each_cnt}'
            )
            
            
    order_lst = [
        'ocsvm',
        'vanilla_dsvdd',
        'smooth_dsvdd',
    ]
    order_lst.extend([
        f'denoising_dsvdd_{noiseRatio}' for noiseRatio in range(1,9)
    ])
    
    result_mean_f = {}
    result_mean_roc = {}
    result_mean_ap = {}
    
    result_std_f = {}
    result_std_roc = {}
    result_std_ap = {}
    
    best_name_f = ''
    best_name_roc = ''
    best_name_ap = ''
    best_score_f = -1
    best_score_roc = -1
    best_score_ap = -1
    
    for order in order_lst:
        
        if np.mean(f_results[order])>= best_score_f:
            best_name_f = order
            best_score_f = np.mean(f_results[order])
            
        if np.mean(roc_results[order]) >= best_score_roc:
            best_name_roc = order
            best_score_roc = np.mean(roc_results[order])
            
        if np.mean(ap_results[order]) >= best_score_ap:
            best_name_ap = order
            best_score_ap = np.mean(ap_results[order])
        
        
        result_mean_f[order] = np.mean(f_results[order])
        result_mean_roc[order] = np.mean(roc_results[order])
        result_mean_ap[order] = np.mean(ap_results[order])
        
        result_std_f[order] = np.std(f_results[order],ddof=1)
        result_std_roc[order] = np.std(roc_results[order],ddof=1)
        result_std_ap[order] = np.std(ap_results[order],ddof=1)
        
    if do_plot:
        f_result_show = []
        roc_result_show = []
        ap_result_show = []
        
        for order in order_lst:
            # print(order)
            f_result_show.append(f_results[order])
            roc_result_show.append(roc_results[order])
            ap_result_show.append(ap_results[order])
            
        # print(f_result_show)
        plt.boxplot(
            f_result_show
        )
        plt.xticks([i+1 for i in range(len(order_lst))],order_lst,rotation=45)
        plt.savefig(
            os.path.join(
                plot_save_path,
                'f1_score.png'
            ),
            dpi = 600
        )
        plt.close()
        plt.cla()
        plt.clf()
        
        
        plt.boxplot(
            roc_result_show
        )
        plt.xticks([i+1 for i in range(len(order_lst))],order_lst,rotation=45)
        plt.savefig(
            os.path.join(
                plot_save_path,
                'roc_auc_score.png'
            ),
            dpi = 600
        )
        plt.close()
        plt.cla()
        plt.clf()
        
        plt.boxplot(
            ap_result_show
        )
        plt.xticks([i+1 for i in range(len(order_lst))],order_lst,rotation=45)
        plt.savefig(
            os.path.join(
                plot_save_path,
                'average_precision.png'
            ),
            dpi = 600
        )
        plt.close()
        plt.cla()
        plt.clf()
    
    return f_results, roc_results, ap_results,result_mean_f,result_mean_roc,result_mean_ap,result_std_f,result_std_roc,result_std_ap,best_name_f,best_name_roc,best_name_ap,best_score_f,best_score_roc,best_score_ap
    
        
import pandas as pd

data_lst = [
    '800_0',
    '800_45'
]

label_lst = ['0.0']+[str(round(0.1*i,1)) for i in range(1,5+1)]

total_result = []

order_lst = [
        'ocsvm',
        'vanilla_dsvdd',
        'smooth_dsvdd',
    ]
order_lst.extend([
    f'denoising_dsvdd_{noiseRatio}' for noiseRatio in range(1,9)
])



df_lst = []
for data in data_lst:
    
    for label in label_lst:
        
        plot_save_path = f'./result_save_path/{data}_{label}'
        os.makedirs(plot_save_path)
        
        f_results, roc_results, ap_results,result_mean_f,result_mean_roc,result_mean_ap,result_std_f,result_std_roc,result_std_ap,best_name_f,best_name_roc,best_name_ap,best_score_f,best_score_roc,best_score_ap = plotPerData(plot_save_path,data,label,do_plot=True)
        
        each_total_result = [
            data,
            label,
            best_name_f,
            best_name_roc,
            best_name_ap,
            best_score_f,
            best_score_roc,
            best_score_ap
        ]
        
        for order in order_lst:
            
            each_total_result.extend(
                [
                    result_mean_f[order],
                    result_std_f[order],
                    result_mean_roc[order],
                    result_std_roc[order],
                    result_mean_ap[order],
                    result_std_ap[order]
                ]
            )
            
        total_result.append(each_total_result)

column_lst = [
    'data_type',
    'normal_label',
    'model achieved highest f1 score(mean)',
    'model achieved highest roc auc score(mean)',
    'model achieved highest average precision score(mean)',
    'highest f1 score(mean)',
    'highest f1 score(mean)',
    'highest f1 score(mean)'
]
for order in order_lst:
    column_lst.extend(
        [
            f'{order}_f1_score_mean',
            f'{order}_f1_score_std',
            f'{order}_roc_auc_mean',
            f'{order}_roc_auc_std',
            f'{order}_average_precision_mean',
            f'{order}_average_precision_std',    
        ]
        
    )
    
total_result = pd.DataFrame(
    total_result,
    columns= column_lst

)
        
        
total_result.to_csv(
    './total_result_testbed.csv',
    index=False
)
        
print('mission complete!!!')        
        
    
        
        



    
        
        
    
    
    
    
    





mission complete!!!


<Figure size 640x480 with 0 Axes>

In [26]:
# import matplotlib.pyplot as plt
# import os
# import yaml
# from pprint import pprint
# import numpy as np



# def open_yaml(yaml_path):
    
#     with open(yaml_path) as f:
#         loadedConfig = yaml.load(f, Loader=yaml.FullLoader)
#     return loadedConfig

# def plotPerData(plot_save_path,data,label,do_plot=True):
    
#     f_results = {
#         f'denoising_dsvdd_{i}':[] for i in range(1,9)
#     }
#     f_results['ocsvm']= []
#     f_results['vanilla_dsvdd'] = []
#     f_results['smooth_dsvdd'] = []
    
#     roc_results= {
#         f'denoising_dsvdd_{i}':[] for i in range(1,9)
#     } 
#     roc_results['ocsvm']= []
#     roc_results['vanilla_dsvdd'] = []
#     roc_results['smooth_dsvdd'] = []
    
#     ap_results = {
#         f'denoising_dsvdd_{i}':[] for i in range(1,9)
#     }
#     ap_results['ocsvm']= []
#     ap_results['vanilla_dsvdd'] = []
#     ap_results['smooth_dsvdd'] = []
    
#     ocsvm_cnt = 0
#     ocsvm_motherPath = f'./history/ocsvm_grad_rawVer_test_testbed_zscore/{data}_noiseRatio_{label}/'
#     for eachCase in os.listdir(ocsvm_motherPath):
#         # print(eachCase)
#         try:
#             eachYamlPath = os.path.join(ocsvm_motherPath,eachCase,'configs/resultConfig.yaml')
            
        
#             openedYaml = open_yaml(eachYamlPath)
            
#             f_results['ocsvm'].append(float(openedYaml['testResult']['f1']))
#             roc_results['ocsvm'].append(float(openedYaml['testResult']['rocAuc']))
#             ap_results['ocsvm'].append(float(openedYaml['testResult']['averagePrecision']))
            
#             ocsvm_cnt +=1
            
#             if ocsvm_cnt == 10:
#                 break
            
#         except:
            
#             pass
            
    
#     vanilla_dsvdd_cnt = 0
#     vanilla_dsvdd_motherPath = f'./history/vanilla_dsvdd_grad_rawVer_test_testbed_zscore/{data}_noiseRatio_{label}/'# ocsvm
#     for eachCase in os.listdir(vanilla_dsvdd_motherPath):
        
#         try:
            
#             eachYamlPath = os.path.join(vanilla_dsvdd_motherPath,eachCase,'configs/resultConfig.yaml')
        
#             openedYaml = open_yaml(eachYamlPath)
#             # pprint(openedYaml)
#             f_results['vanilla_dsvdd'].append(float(openedYaml['testResult']['f1_score']))
#             roc_results['vanilla_dsvdd'].append(float(openedYaml['testResult']['roc_auc']))
#             ap_results['vanilla_dsvdd'].append(float(openedYaml['testResult']['average_precision']))
            
#             vanilla_dsvdd_cnt += 1
            
#             if vanilla_dsvdd_cnt == 10:
#                 break
        
#         except:
            
#             pass
            
#     smooth_dsvdd_cnt = 0
#     smooth_dsvdd_motherPath = f'./history/smoothed_dsvdd_grad_rawVer_test_testbed_zscore/{data}_noiseRatio_{label}/'# ocsvm
#     for eachCase in os.listdir(smooth_dsvdd_motherPath):
        
#         try:
    
#             eachYamlPath = os.path.join(smooth_dsvdd_motherPath,eachCase,'configs/resultConfig.yaml')
        
#             openedYaml = open_yaml(eachYamlPath)
        
#             f_results['smooth_dsvdd'].append(float(openedYaml['testResult']['f1_score']))
#             roc_results['smooth_dsvdd'].append(float(openedYaml['testResult']['roc_auc']))
#             ap_results['smooth_dsvdd'].append(float(openedYaml['testResult']['average_precision']))
            
#             smooth_dsvdd_cnt += 1
            
#             if smooth_dsvdd_cnt == 10:
#                 break
            
#         except:
            
#             pass
    
    
#     denosing_cnt_lst = []
#     denoising_motherPath = f'./history/denoising_dsvdd_grad_rawVer_test_testbed_zscore/{data}_noiseRatio_{label}/'    
#     for noiseRatio in range(1,9):
        
#         eachMotherPath = os.path.join(denoising_motherPath,'noise_'+str(round(0.1*noiseRatio,1)))
        
#         denoising_cnt = 0
        
#         for eachCase in os.listdir(eachMotherPath):
            
#             try:
#                 eachYamlPath = os.path.join(eachMotherPath,eachCase,'configs/resultConfig.yaml')
            
#                 openedYaml = open_yaml(eachYamlPath)
            
#                 f_results[f'denoising_dsvdd_{noiseRatio}'].append(float(openedYaml['testResult']['f1_score']))
#                 roc_results[f'denoising_dsvdd_{noiseRatio}'].append(float(openedYaml['testResult']['roc_auc']))
#                 ap_results[f'denoising_dsvdd_{noiseRatio}'].append(float(openedYaml['testResult']['average_precision']))
                
#                 denoising_cnt += 1
                
#                 if denoising_cnt == 10:
#                     break
                
#             except:
                
#                 pass
        
#         denosing_cnt_lst.append(denoising_cnt)
                
    
#     if ocsvm_cnt != 10:
#         print(f'{data,label}ocsvm_cnt must be 10 but got {ocsvm_cnt}')
#         raise Exception(
#             f'ocsvm_cnt must be 10 but got {ocsvm_cnt}'
#         )
    
#     if vanilla_dsvdd_cnt != 10:
        
#         print(f'{data,label}vanilla_cnt must be 10 but got {vanilla_dsvdd_cnt}')
#         raise Exception(
#             f'vanilla_cnt must be 10 but got {vanilla_dsvdd_cnt}'
#         )
        
#     if smooth_dsvdd_cnt != 10:
        
#         print(f'{data,label}smooth_cnt must be 10 but got {smooth_dsvdd_cnt}')
#         raise Exception(
            
#             f'smooth_cnt must be 10 but got {smooth_dsvdd_cnt}'
#         )
        
#     for idx,each_cnt in enumerate(denosing_cnt_lst):
        
#         if each_cnt != 10:
#             print(f'{data,label}noise ratio {idx+1} cnt must be 10 but got {each_cnt}')
#             raise Exception(
#                 f'noise ratio {idx+1} cnt must be 10 but got {each_cnt}'
#             )
            
            
#     order_lst = [
#         'ocsvm',
#         'vanilla_dsvdd',
#         'smooth_dsvdd',
#     ]
#     order_lst.extend([
#         f'denoising_dsvdd_{noiseRatio}' for noiseRatio in range(1,9)
#     ])
    
#     result_mean_f = {}
#     result_mean_roc = {}
#     result_mean_ap = {}
    
#     result_std_f = {}
#     result_std_roc = {}
#     result_std_ap = {}
    
#     best_name_f = ''
#     best_name_roc = ''
#     best_name_ap = ''
#     best_score_f = -1
#     best_score_roc = -1
#     best_score_ap = -1
    
#     for order in order_lst:
        
#         if np.mean(f_results[order])>= best_score_f:
#             best_name_f = order
#             best_score_f = np.mean(f_results[order])
            
#         if np.mean(roc_results[order]) >= best_score_roc:
#             best_name_roc = order
#             best_score_roc = np.mean(roc_results[order])
            
#         if np.mean(ap_results[order]) >= best_score_ap:
#             best_name_ap = order
#             best_score_ap = np.mean(ap_results[order])
        
        
#         result_mean_f[order] = np.mean(f_results[order])
#         result_mean_roc[order] = np.mean(roc_results[order])
#         result_mean_ap[order] = np.mean(ap_results[order])
        
#         result_std_f[order] = np.std(f_results[order],ddof=1)
#         result_std_roc[order] = np.std(roc_results[order],ddof=1)
#         result_std_ap[order] = np.std(ap_results[order],ddof=1)
        
#     if do_plot:
#         f_result_show = []
#         roc_result_show = []
#         ap_result_show = []
        
#         for order in order_lst:
#             # print(order)
#             f_result_show.append(f_results[order])
#             roc_result_show.append(roc_results[order])
#             ap_result_show.append(ap_results[order])
            
#         # print(f_result_show)
#         plt.boxplot(
#             f_result_show
#         )
#         plt.xticks([i+1 for i in range(len(order_lst))],order_lst,rotation=45)
#         plt.savefig(
#             os.path.join(
#                 plot_save_path,
#                 'f1_score.png'
#             )
#         )
#         plt.close()
#         plt.cla()
#         plt.clf()
        
        
#         plt.boxplot(
#             roc_result_show
#         )
#         plt.xticks([i+1 for i in range(len(order_lst))],order_lst,rotation=45)
#         plt.savefig(
#             os.path.join(
#                 plot_save_path,
#                 'roc_auc_score.png'
#             )
#         )
#         plt.close()
#         plt.cla()
#         plt.clf()
        
#         plt.boxplot(
#             ap_result_show
#         )
#         plt.xticks([i+1 for i in range(len(order_lst))],order_lst,rotation=45)
#         plt.savefig(
#             os.path.join(
#                 plot_save_path,
#                 'average_precision.png'
#             )
#         )
#         plt.close()
#         plt.cla()
#         plt.clf()
    
#     return f_results, roc_results, ap_results,result_mean_f,result_mean_roc,result_mean_ap,result_std_f,result_std_roc,result_std_ap,best_name_f,best_name_roc,best_name_ap,best_score_f,best_score_roc,best_score_ap
    
        
# import pandas as pd

# data_lst = [
#     # '800_0',
#     '800_45'
# ]

# label_lst = ['0.0']+[str(round(0.1*i,1)) for i in range(1,5+1)]

# total_result = []

# order_lst = [
#         'ocsvm',
#         'vanilla_dsvdd',
#         'smooth_dsvdd',
#     ]
# order_lst.extend([
#     f'denoising_dsvdd_{noiseRatio}' for noiseRatio in range(1,9)
# ])



# df_lst = []
# for data in data_lst:
    
#     for label in label_lst:
        
#         plot_save_path = f'./result_save_path/{data}_{label}_normalized'
#         os.makedirs(plot_save_path)
        
#         f_results, roc_results, ap_results,result_mean_f,result_mean_roc,result_mean_ap,result_std_f,result_std_roc,result_std_ap,best_name_f,best_name_roc,best_name_ap,best_score_f,best_score_roc,best_score_ap = plotPerData(plot_save_path,data,label,do_plot=True)
        
#         each_total_result = [
#             data,
#             label,
#             best_name_f,
#             best_name_roc,
#             best_name_ap,
#             best_score_f,
#             best_score_roc,
#             best_score_ap
#         ]
        
#         for order in order_lst:
            
#             each_total_result.extend(
#                 [
#                     result_mean_f[order],
#                     result_std_f[order],
#                     result_mean_roc[order],
#                     result_std_roc[order],
#                     result_mean_ap[order],
#                     result_std_ap[order]
#                 ]
#             )
            
#         total_result.append(each_total_result)

# column_lst = [
#     'data_type',
#     'normal_label',
#     'model achieved highest f1 score(mean)',
#     'model achieved highest roc auc score(mean)',
#     'model achieved highest average precision score(mean)',
#     'highest f1 score(mean)',
#     'highest f1 score(mean)',
#     'highest f1 score(mean)'
# ]
# for order in order_lst:
#     column_lst.extend(
#         [
#             f'{order}_f1_score_mean',
#             f'{order}_f1_score_std',
#             f'{order}_roc_auc_mean',
#             f'{order}_roc_auc_std',
#             f'{order}_average_precision_mean',
#             f'{order}_average_precision_std',    
#         ]
        
#     )
    
# total_result = pd.DataFrame(
#     total_result,
#     columns= column_lst

# )
        
        
# total_result.to_csv(
#     './total_result_testbed_normalized.csv',
#     index=False
# )
        
# print('mission complete!!!')        
        
    
        
        



    
        
        
    
    
    
    
    





mission complete!!!


<Figure size 640x480 with 0 Axes>

In [21]:
import matplotlib.pyplot as plt
import os
import yaml
from pprint import pprint
import numpy as np



def open_yaml(yaml_path):
    
    with open(yaml_path) as f:
        loadedConfig = yaml.load(f, Loader=yaml.FullLoader)
    return loadedConfig

def plotPerData(plot_save_path,data,label,do_plot=True):
    
    f_results = {
        f'denoising_dsvdd_{i}':[] for i in range(1,9)
    }
    f_results['ocsvm']= []
    f_results['vanilla_dsvdd'] = []
    f_results['smooth_dsvdd'] = []
    
    roc_results= {
        f'denoising_dsvdd_{i}':[] for i in range(1,9)
    } 
    roc_results['ocsvm']= []
    roc_results['vanilla_dsvdd'] = []
    roc_results['smooth_dsvdd'] = []
    
    ap_results = {
        f'denoising_dsvdd_{i}':[] for i in range(1,9)
    }
    ap_results['ocsvm']= []
    ap_results['vanilla_dsvdd'] = []
    ap_results['smooth_dsvdd'] = []
    
    ocsvm_cnt = 0
    ocsvm_motherPath = f'./history/ocsvm_grad_FEed_test_testbed/{data}_noiseRatio_{label}/'
    for eachCase in os.listdir(ocsvm_motherPath):
        # print(eachCase)
        try:
            eachYamlPath = os.path.join(ocsvm_motherPath,eachCase,'configs/resultConfig.yaml')
            
        
            openedYaml = open_yaml(eachYamlPath)
            
            f_results['ocsvm'].append(float(openedYaml['testResult']['f1']))
            roc_results['ocsvm'].append(float(openedYaml['testResult']['rocAuc']))
            ap_results['ocsvm'].append(float(openedYaml['testResult']['averagePrecision']))
            
            ocsvm_cnt +=1
            
            if ocsvm_cnt == 10:
                break
            
        except:
            
            pass
            
    
    vanilla_dsvdd_cnt = 0
    vanilla_dsvdd_motherPath = f'./history/vanilla_dsvdd_grad_FEed_test_testbed/{data}_noiseRatio_{label}/'# ocsvm
    for eachCase in os.listdir(vanilla_dsvdd_motherPath):
        
        try:
            
            eachYamlPath = os.path.join(vanilla_dsvdd_motherPath,eachCase,'configs/resultConfig.yaml')
        
            openedYaml = open_yaml(eachYamlPath)
            # pprint(openedYaml)
            f_results['vanilla_dsvdd'].append(float(openedYaml['testResult']['f1_score']))
            roc_results['vanilla_dsvdd'].append(float(openedYaml['testResult']['roc_auc']))
            ap_results['vanilla_dsvdd'].append(float(openedYaml['testResult']['average_precision']))
            
            vanilla_dsvdd_cnt += 1
            
            if vanilla_dsvdd_cnt == 10:
                break
        
        except:
            
            pass
        
    smooth_dsvdd_cnt = 0
    smooth_dsvdd_motherPath = f'./history/smoothed_dsvdd_grad_FEed_test_testbed/{data}_noiseRatio_{label}/'# ocsvm
    for eachCase in os.listdir(smooth_dsvdd_motherPath):
        
        try:
    
            eachYamlPath = os.path.join(smooth_dsvdd_motherPath,eachCase,'configs/resultConfig.yaml')
        
            openedYaml = open_yaml(eachYamlPath)
        
            f_results['smooth_dsvdd'].append(float(openedYaml['testResult']['f1_score']))
            roc_results['smooth_dsvdd'].append(float(openedYaml['testResult']['roc_auc']))
            ap_results['smooth_dsvdd'].append(float(openedYaml['testResult']['average_precision']))
            
            smooth_dsvdd_cnt += 1
            
            if smooth_dsvdd_cnt == 10:
                break
            
        except:
            
            pass
    
    denosing_cnt_lst = []
    denoising_motherPath = f'./history/denoising_dsvdd_grad_FEed_test_testbed/{data}_noiseRatio_{label}/'    
    for noiseRatio in range(1,9):
        
        eachMotherPath = os.path.join(denoising_motherPath,'noise_'+str(round(0.1*noiseRatio,1)))
        
        denoising_cnt = 0
        
        for eachCase in os.listdir(eachMotherPath):
            
            try:
                eachYamlPath = os.path.join(eachMotherPath,eachCase,'configs/resultConfig.yaml')
            
                openedYaml = open_yaml(eachYamlPath)
            
                f_results[f'denoising_dsvdd_{noiseRatio}'].append(float(openedYaml['testResult']['f1_score']))
                roc_results[f'denoising_dsvdd_{noiseRatio}'].append(float(openedYaml['testResult']['roc_auc']))
                ap_results[f'denoising_dsvdd_{noiseRatio}'].append(float(openedYaml['testResult']['average_precision']))
                
                denoising_cnt += 1
                
                if denoising_cnt == 10:
                    break
                
            except:
                
                pass
        
        denosing_cnt_lst.append(denoising_cnt)
                
    
    if ocsvm_cnt != 10:
        print(f'{data,label}ocsvm_cnt must be 10 but got {ocsvm_cnt}')
        raise Exception(
            f'ocsvm_cnt must be 10 but got {ocsvm_cnt}'
        )
    
    if vanilla_dsvdd_cnt != 10:
        
        print(f'{data,label}vanilla_cnt must be 10 but got {vanilla_dsvdd_cnt}')
        raise Exception(
            f'vanilla_cnt must be 10 but got {vanilla_dsvdd_cnt}'
        )
        
    if smooth_dsvdd_cnt != 10:
        
        print(f'{data,label}smooth_cnt must be 10 but got {smooth_dsvdd_cnt}')
        raise Exception(
            
            f'smooth_cnt must be 10 but got {smooth_dsvdd_cnt}'
        )
        
    for idx,each_cnt in enumerate(denosing_cnt_lst):
        
        if each_cnt != 10:
            print(f'{data,label}noise ratio {idx+1} cnt must be 10 but got {each_cnt}')
            raise Exception(
                f'noise ratio {idx+1} cnt must be 10 but got {each_cnt}'
            )
            
            
    order_lst = [
        'ocsvm',
        'vanilla_dsvdd',
        'smooth_dsvdd',
    ]
    order_lst.extend([
        f'denoising_dsvdd_{noiseRatio}' for noiseRatio in range(1,9)
    ])
    
    result_mean_f = {}
    result_mean_roc = {}
    result_mean_ap = {}
    
    result_std_f = {}
    result_std_roc = {}
    result_std_ap = {}
    
    best_name_f = ''
    best_name_roc = ''
    best_name_ap = ''
    best_score_f = -1
    best_score_roc = -1
    best_score_ap = -1
    
    for order in order_lst:
        
        if np.mean(f_results[order])>= best_score_f:
            best_name_f = order
            best_score_f = np.mean(f_results[order])
            
        if np.mean(roc_results[order]) >= best_score_roc:
            best_name_roc = order
            best_score_roc = np.mean(roc_results[order])
            
        if np.mean(ap_results[order]) >= best_score_ap:
            best_name_ap = order
            best_score_ap = np.mean(ap_results[order])
        
        
        result_mean_f[order] = np.mean(f_results[order])
        result_mean_roc[order] = np.mean(roc_results[order])
        result_mean_ap[order] = np.mean(ap_results[order])
        
        result_std_f[order] = np.std(f_results[order],ddof=1)
        result_std_roc[order] = np.std(roc_results[order],ddof=1)
        result_std_ap[order] = np.std(ap_results[order],ddof=1)
        
    if do_plot:
        f_result_show = []
        roc_result_show = []
        ap_result_show = []
        
        for order in order_lst:
            # print(order)
            f_result_show.append(f_results[order])
            roc_result_show.append(roc_results[order])
            ap_result_show.append(ap_results[order])
            
        # print(f_result_show)
        plt.boxplot(
            f_result_show
        )
        plt.xticks([i+1 for i in range(len(order_lst))],order_lst,rotation=45)
        plt.savefig(
            os.path.join(
                plot_save_path,
                'f1_score.png'
            )
        )
        plt.close()
        plt.cla()
        plt.clf()
        
        
        plt.boxplot(
            roc_result_show
        )
        plt.xticks([i+1 for i in range(len(order_lst))],order_lst,rotation=45)
        plt.savefig(
            os.path.join(
                plot_save_path,
                'roc_auc_score.png'
            )
        )
        plt.close()
        plt.cla()
        plt.clf()
        
        plt.boxplot(
            ap_result_show
        )
        plt.xticks([i+1 for i in range(len(order_lst))],order_lst,rotation=45)
        plt.savefig(
            os.path.join(
                plot_save_path,
                'average_precision.png'
            )
        )
        plt.close()
        plt.cla()
        plt.clf()
    
    return f_results, roc_results, ap_results,result_mean_f,result_mean_roc,result_mean_ap,result_std_f,result_std_roc,result_std_ap,best_name_f,best_name_roc,best_name_ap,best_score_f,best_score_roc,best_score_ap
    
        
import pandas as pd

data_lst = [
    'FEed_800_0',
    'FEed_800_45'
]

label_lst = ['0.0']+[str(round(0.1*i,1)) for i in range(1,5+1)]

total_result = []

order_lst = [
        'ocsvm',
        'vanilla_dsvdd',
        'smooth_dsvdd',
    ]
order_lst.extend([
    f'denoising_dsvdd_{noiseRatio}' for noiseRatio in range(1,9)
])



df_lst = []
for data in data_lst:
    
    for label in label_lst:
        
        plot_save_path = f'./result_save_path/{data}_{label}_FEed'
        os.makedirs(plot_save_path)
        
        f_results, roc_results, ap_results,result_mean_f,result_mean_roc,result_mean_ap,result_std_f,result_std_roc,result_std_ap,best_name_f,best_name_roc,best_name_ap,best_score_f,best_score_roc,best_score_ap = plotPerData(plot_save_path,data,label,do_plot=True)
        
        each_total_result = [
            data,
            label,
            best_name_f,
            best_name_roc,
            best_name_ap,
            best_score_f,
            best_score_roc,
            best_score_ap
        ]
        
        for order in order_lst:
            
            each_total_result.extend(
                [
                    result_mean_f[order],
                    result_std_f[order],
                    result_mean_roc[order],
                    result_std_roc[order],
                    result_mean_ap[order],
                    result_std_ap[order]
                ]
            )
            
        total_result.append(each_total_result)

column_lst = [
    'data_type',
    'normal_label',
    'model achieved highest f1 score(mean)',
    'model achieved highest roc auc score(mean)',
    'model achieved highest average precision score(mean)',
    'highest f1 score(mean)',
    'highest f1 score(mean)',
    'highest f1 score(mean)'
]
for order in order_lst:
    column_lst.extend(
        [
            f'{order}_f1_score_mean',
            f'{order}_f1_score_std',
            f'{order}_roc_auc_mean',
            f'{order}_roc_auc_std',
            f'{order}_average_precision_mean',
            f'{order}_average_precision_std',    
        ]
        
    )
    
total_result = pd.DataFrame(
    total_result,
    columns= column_lst

)
        
        
total_result.to_csv(
    './total_result_testbed_FEed.csv',
    index=False
)
        
print('mission complete!!!')        
        
    
        
        



    
        
        
    
    
    
    
    





mission complete!!!


<Figure size 640x480 with 0 Axes>

In [22]:
import matplotlib.pyplot as plt
import os
import yaml
from pprint import pprint
import numpy as np



def open_yaml(yaml_path):
    
    with open(yaml_path) as f:
        loadedConfig = yaml.load(f, Loader=yaml.FullLoader)
    return loadedConfig

def plotPerData(plot_save_path,data,label,do_plot=True):
    
    f_results = {
        f'denoising_dsvdd_{i}':[] for i in range(1,9)
    }
    f_results['ocsvm']= []
    f_results['vanilla_dsvdd'] = []
    f_results['smooth_dsvdd'] = []
    
    roc_results= {
        f'denoising_dsvdd_{i}':[] for i in range(1,9)
    } 
    roc_results['ocsvm']= []
    roc_results['vanilla_dsvdd'] = []
    roc_results['smooth_dsvdd'] = []
    
    ap_results = {
        f'denoising_dsvdd_{i}':[] for i in range(1,9)
    }
    ap_results['ocsvm']= []
    ap_results['vanilla_dsvdd'] = []
    ap_results['smooth_dsvdd'] = []
    
    ocsvm_cnt = 0
    ocsvm_motherPath = f'./history/ocsvm_grad_FEedVer_test_testbed_zscore/{data}_noiseRatio_{label}/'
    for eachCase in os.listdir(ocsvm_motherPath):
        # print(eachCase)
        try:
            eachYamlPath = os.path.join(ocsvm_motherPath,eachCase,'configs/resultConfig.yaml')
            
        
            openedYaml = open_yaml(eachYamlPath)
            
            f_results['ocsvm'].append(float(openedYaml['testResult']['f1']))
            roc_results['ocsvm'].append(float(openedYaml['testResult']['rocAuc']))
            ap_results['ocsvm'].append(float(openedYaml['testResult']['averagePrecision']))
            
            ocsvm_cnt +=1
            
            if ocsvm_cnt == 10:
                break
            
        except:
            
            pass
            
    
    vanilla_dsvdd_cnt = 0
    vanilla_dsvdd_motherPath = f'./history/vanilla_dsvdd_grad_FEedVer_test_testbed_zscore/{data}_noiseRatio_{label}/'# ocsvm
    for eachCase in os.listdir(vanilla_dsvdd_motherPath):
        
        try:
            
            eachYamlPath = os.path.join(vanilla_dsvdd_motherPath,eachCase,'configs/resultConfig.yaml')
        
            openedYaml = open_yaml(eachYamlPath)
            # pprint(openedYaml)
            f_results['vanilla_dsvdd'].append(float(openedYaml['testResult']['f1_score']))
            roc_results['vanilla_dsvdd'].append(float(openedYaml['testResult']['roc_auc']))
            ap_results['vanilla_dsvdd'].append(float(openedYaml['testResult']['average_precision']))
            
            vanilla_dsvdd_cnt += 1
            
            if vanilla_dsvdd_cnt == 10:
                break
        
        except:
            
            pass
        
    smooth_dsvdd_cnt = 0
    smooth_dsvdd_motherPath = f'./history/smoothed_dsvdd_grad_FEedVer_test_testbed_zscore/{data}_noiseRatio_{label}/'# ocsvm
    for eachCase in os.listdir(smooth_dsvdd_motherPath):
        
        try:
    
            eachYamlPath = os.path.join(smooth_dsvdd_motherPath,eachCase,'configs/resultConfig.yaml')
        
            openedYaml = open_yaml(eachYamlPath)
        
            f_results['smooth_dsvdd'].append(float(openedYaml['testResult']['f1_score']))
            roc_results['smooth_dsvdd'].append(float(openedYaml['testResult']['roc_auc']))
            ap_results['smooth_dsvdd'].append(float(openedYaml['testResult']['average_precision']))
            
            smooth_dsvdd_cnt += 1
            
            if smooth_dsvdd_cnt == 10:
                break
            
        except:
            
            pass
    
    denosing_cnt_lst = []
    denoising_motherPath = f'./history/denoising_dsvdd_grad_FEedVer_test_testbed_zscore/{data}_noiseRatio_{label}/'    
    for noiseRatio in range(1,9):
        
        eachMotherPath = os.path.join(denoising_motherPath,'noise_'+str(round(0.1*noiseRatio,1)))
        
        denoising_cnt = 0
        
        for eachCase in os.listdir(eachMotherPath):
            
            try:
                eachYamlPath = os.path.join(eachMotherPath,eachCase,'configs/resultConfig.yaml')
            
                openedYaml = open_yaml(eachYamlPath)
            
                f_results[f'denoising_dsvdd_{noiseRatio}'].append(float(openedYaml['testResult']['f1_score']))
                roc_results[f'denoising_dsvdd_{noiseRatio}'].append(float(openedYaml['testResult']['roc_auc']))
                ap_results[f'denoising_dsvdd_{noiseRatio}'].append(float(openedYaml['testResult']['average_precision']))
                
                denoising_cnt += 1
                
                if denoising_cnt == 10:
                    break
                
            except:
                
                pass
        
        denosing_cnt_lst.append(denoising_cnt)
                
    
    if ocsvm_cnt != 10:
        print(f'{data,label}ocsvm_cnt must be 10 but got {ocsvm_cnt}')
        raise Exception(
            f'ocsvm_cnt must be 10 but got {ocsvm_cnt}'
        )
    
    if vanilla_dsvdd_cnt != 10:
        
        print(f'{data,label}vanilla_cnt must be 10 but got {vanilla_dsvdd_cnt}')
        raise Exception(
            f'vanilla_cnt must be 10 but got {vanilla_dsvdd_cnt}'
        )
        
    if smooth_dsvdd_cnt != 10:
        
        print(f'{data,label}smooth_cnt must be 10 but got {smooth_dsvdd_cnt}')
        raise Exception(
            
            f'smooth_cnt must be 10 but got {smooth_dsvdd_cnt}'
        )
        
    for idx,each_cnt in enumerate(denosing_cnt_lst):
        
        if each_cnt != 10:
            print(f'{data,label}noise ratio {idx+1} cnt must be 10 but got {each_cnt}')
            raise Exception(
                f'noise ratio {idx+1} cnt must be 10 but got {each_cnt}'
            )
            
            
    order_lst = [
        'ocsvm',
        'vanilla_dsvdd',
        'smooth_dsvdd',
    ]
    order_lst.extend([
        f'denoising_dsvdd_{noiseRatio}' for noiseRatio in range(1,9)
    ])
    
    result_mean_f = {}
    result_mean_roc = {}
    result_mean_ap = {}
    
    result_std_f = {}
    result_std_roc = {}
    result_std_ap = {}
    
    best_name_f = ''
    best_name_roc = ''
    best_name_ap = ''
    best_score_f = -1
    best_score_roc = -1
    best_score_ap = -1
    
    for order in order_lst:
        
        if np.mean(f_results[order])>= best_score_f:
            best_name_f = order
            best_score_f = np.mean(f_results[order])
            
        if np.mean(roc_results[order]) >= best_score_roc:
            best_name_roc = order
            best_score_roc = np.mean(roc_results[order])
            
        if np.mean(ap_results[order]) >= best_score_ap:
            best_name_ap = order
            best_score_ap = np.mean(ap_results[order])
        
        
        result_mean_f[order] = np.mean(f_results[order])
        result_mean_roc[order] = np.mean(roc_results[order])
        result_mean_ap[order] = np.mean(ap_results[order])
        
        result_std_f[order] = np.std(f_results[order],ddof=1)
        result_std_roc[order] = np.std(roc_results[order],ddof=1)
        result_std_ap[order] = np.std(ap_results[order],ddof=1)
        
    if do_plot:
        f_result_show = []
        roc_result_show = []
        ap_result_show = []
        
        for order in order_lst:
            # print(order)
            f_result_show.append(f_results[order])
            roc_result_show.append(roc_results[order])
            ap_result_show.append(ap_results[order])
            
        # print(f_result_show)
        plt.boxplot(
            f_result_show
        )
        plt.xticks([i+1 for i in range(len(order_lst))],order_lst,rotation=45)
        plt.savefig(
            os.path.join(
                plot_save_path,
                'f1_score.png'
            )
        )
        plt.close()
        plt.cla()
        plt.clf()
        
        
        plt.boxplot(
            roc_result_show
        )
        plt.xticks([i+1 for i in range(len(order_lst))],order_lst,rotation=45)
        plt.savefig(
            os.path.join(
                plot_save_path,
                'roc_auc_score.png'
            )
        )
        plt.close()
        plt.cla()
        plt.clf()
        
        plt.boxplot(
            ap_result_show
        )
        plt.xticks([i+1 for i in range(len(order_lst))],order_lst,rotation=45)
        plt.savefig(
            os.path.join(
                plot_save_path,
                'average_precision.png'
            )
        )
        plt.close()
        plt.cla()
        plt.clf()
    
    return f_results, roc_results, ap_results,result_mean_f,result_mean_roc,result_mean_ap,result_std_f,result_std_roc,result_std_ap,best_name_f,best_name_roc,best_name_ap,best_score_f,best_score_roc,best_score_ap
    
        
import pandas as pd

data_lst = [
    'FEed_800_0',
    'FEed_800_45'
]

label_lst = ['0.0']+[str(round(0.1*i,1)) for i in range(1,5+1)]

total_result = []

order_lst = [
        'ocsvm',
        'vanilla_dsvdd',
        'smooth_dsvdd',
    ]
order_lst.extend([
    f'denoising_dsvdd_{noiseRatio}' for noiseRatio in range(1,9)
])



df_lst = []
for data in data_lst:
    
    for label in label_lst:
        
        plot_save_path = f'./result_save_path/{data}_{label}_FEed_normalized'
        os.makedirs(plot_save_path)
        
        f_results, roc_results, ap_results,result_mean_f,result_mean_roc,result_mean_ap,result_std_f,result_std_roc,result_std_ap,best_name_f,best_name_roc,best_name_ap,best_score_f,best_score_roc,best_score_ap = plotPerData(plot_save_path,data,label,do_plot=True)
        
        each_total_result = [
            data,
            label,
            best_name_f,
            best_name_roc,
            best_name_ap,
            best_score_f,
            best_score_roc,
            best_score_ap
        ]
        
        for order in order_lst:
            
            each_total_result.extend(
                [
                    result_mean_f[order],
                    result_std_f[order],
                    result_mean_roc[order],
                    result_std_roc[order],
                    result_mean_ap[order],
                    result_std_ap[order]
                ]
            )
            
        total_result.append(each_total_result)

column_lst = [
    'data_type',
    'normal_label',
    'model achieved highest f1 score(mean)',
    'model achieved highest roc auc score(mean)',
    'model achieved highest average precision score(mean)',
    'highest f1 score(mean)',
    'highest f1 score(mean)',
    'highest f1 score(mean)'
]
for order in order_lst:
    column_lst.extend(
        [
            f'{order}_f1_score_mean',
            f'{order}_f1_score_std',
            f'{order}_roc_auc_mean',
            f'{order}_roc_auc_std',
            f'{order}_average_precision_mean',
            f'{order}_average_precision_std',    
        ]
        
    )
    
total_result = pd.DataFrame(
    total_result,
    columns= column_lst

)
        
        
total_result.to_csv(
    './total_result_testbed_FEed_normalized.csv',
    index=False
)
        
print('mission complete!!!')        
        
    
        
        



    
        
        
    
    
    
    
    





mission complete!!!


<Figure size 640x480 with 0 Axes>